In [ ]:
#@title Install
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "transformers==4.57.3" accelerate
import importlib.metadata as _md
for p in ('transformers','torch','accelerate'):
    print(f'  {p:14}{_md.version(p)}')


In [ ]:
#@title Configuration
import os, sys, json, re, time, gc
from pathlib import Path
from collections import defaultdict, Counter
import torch

ROOT = Path('/content/drive/MyDrive/bearing_protocol')
RUNS_DIR=ROOT/'runs'; EVAL_DIR=ROOT/'eval'
CACHE=EVAL_DIR/'judge_cache'; CACHE.mkdir(parents=True, exist_ok=True)
DATA_DIRS={'PB':ROOT/'data_pb_5khz', 'CW':ROOT/'data_cw_5khz'}
SPEC={'PB':'spec_v4.json','CW':'spec_cw_v2.json'}

JUDGE_MODEL='Qwen/Qwen2.5-32B-Instruct'
JUDGE_BATCH=8
JUDGE_MAX_NEW=24
ALL_CONDS=['correct','contrastive','blind','substituted']
VERDICT_CONDS=['correct','contrastive']

free_gb=torch.cuda.mem_get_info()[0]/2**30
print(f'GPU {torch.cuda.get_device_name(0)}, {free_gb:.0f} GB free')
if free_gb < 66+12: print('warning: less than 78 GB of GPU memory free')

RUNS=[]
for p in sorted(RUNS_DIR.glob('*.json')):
    r=json.load(open(p))
    if r.get('status')!='done' or not r.get('result_dir') or not r.get('subsets'): continue
    d=Path(r['result_dir'])
    conds=[c for c in ALL_CONDS if (d/f'infer_{c}.jsonl').exists()]
    if conds:
        RUNS.append({'rid':r['run_id'],'dir':d,'conds':conds,'dataset':r['dataset'],'label':r['model'],
                     'modality':r['modality'],'ratio':r['ratio'],'seed':r['seed'],'subsets':r['subsets']})
print(f'\n{len(RUNS)} runs with generations')
for r in RUNS: print(f"   {r['rid']:44} {', '.join(r['conds'])}")

In [ ]:
#@title Parser
TH_BY={ds:json.load(open(DATA_DIRS[ds]/SPEC[ds]))['thresholds'] for ds in DATA_DIRS if (DATA_DIRS[ds]/SPEC[ds]).exists()}
CLASSES=['Healthy','Inner_Ring_Fault','Outer_Ring_Fault']
PRETTY_TO_CLASS={'healthy bearing':'Healthy','inner ring fault':'Inner_Ring_Fault',
                 'outer ring fault':'Outer_Ring_Fault'}
CLAIM_PATTERNS={
 'env_char_energy_frac':[(r'only a small fraction of the envelope energy','low'),
   (r'a moderate fraction of the envelope energy','mid'),
   (r'a large fraction of the envelope energy','high')],
 'env_bpfo_ratio':[(r'little energy at the outer-race','low'),
   (r'moderate energy at the outer-race','mid'),
   (r'strongly concentrated at the outer-race','high')],
 'env_bpfo_harmonic_ratio':[(r'no harmonic banding','low'),
   (r'weak harmonic content','mid'),(r'clear harmonic banding','high')],
 'env_bpfi_ratio':[(r'little energy at the inner-race','low'),
   (r'moderate energy at the inner-race','mid'),
   (r'strongly concentrated at the inner-race','high')],
 'env_bpfi_sideband_ratio':[(r'no shaft-rate sideband','low'),
   (r'weak shaft-rate sideband','mid'),(r'clear shaft-rate sideband','high')],
}
QUANTS=[('env_char_energy_frac',
  'how much of the signal energy sits at the bearing characteristic frequencies '
  'rather than spread across the band (the text calls this a small, moderate or '
  'large fraction of the energy)'),
 ('env_bpfo_ratio','the energy at the outer-race passing rate'),
 ('env_bpfo_harmonic_ratio',
  'harmonic banding at twice and three times the OUTER-race rate (the text says '
  'no harmonic banding is visible, or weak harmonic content is present, or clear '
  'harmonic banding is present)'),
 ('env_bpfi_ratio',
  'the amount of energy at the INNER-race passing rate specifically, not the '
  'outer-race rate (the text says there is little, moderate, or strongly '
  'concentrated energy there)'),
 ('env_bpfi_sideband_ratio',
  'the shaft-rate sideband below the INNER-race rate (the text says no '
  'shaft-rate sideband is present, or a weak one, or a clear one)')]

def det_extract(text):
    t=re.sub(r'<think>.*?</think>',' ',(text or ''),flags=re.S|re.I)
    m=re.search(r'<explanation>(.*?)</explanation>', t, re.S|re.I)
    body=m.group(1) if m else t
    low=body.lower()
    mm=re.search(r'consistent with (?:a|an)\s+(healthy bearing|inner ring fault|'
                 r'outer ring fault)', low)
    return {'class':PRETTY_TO_CLASS[mm.group(1)] if mm else None,
            'rejected':bool(re.search(r'however,\s*the cnn classified this as',low)),
            'claims':{q:next((l for p,l in pats if re.search(p,low)),None)
                      for q,pats in CLAIM_PATTERNS.items()},
            'body':body}

def get_response(row):
    for k in ('response','generated_text','completion'):
        if isinstance(row.get(k),str) and row[k]: return row[k]
    m=row.get('messages')
    return m[-1].get('content','') if m and m[-1].get('role')=='assistant' else ''
def load_jsonl(p):
    with open(p) as f: return [json.loads(l) for l in f if l.strip()]
def true_level(q,v,ds='PB'):
    lo,hi=TH_BY[ds][q]; return 'low' if v<lo else ('high' if v>=hi else 'mid')
DATA={}
for r in RUNS:
    for cond in r['conds']:
        sub=Path(r['subsets']['blind' if cond=='substituted' else cond])
        meta=load_jsonl(sub); gen=load_jsonl(r['dir']/f'infer_{cond}.jsonl')
        if len(meta)!=len(gen):
            print(f"  ! {r['rid']}/{cond}: {len(meta)} meta vs {len(gen)} generations, skipping"); continue
        DATA[(r['rid'],cond)]=[{**m['meta'],**det_extract(get_response(g)),'dataset':r['dataset']}
                               for m,g in zip(meta,gen)]
print(f'loaded {len(DATA)} run/condition pairs')

In [ ]:
#@title Judge prompts
JUDGE_SYS=('You read a bearing-diagnosis explanation and report what it says. '
 'You never judge whether it is correct. You answer with one word only.')

LEVEL_TMPL=(
 'Explanation:\n"""\n{body}\n"""\n\n'
 'Question: what does this explanation say about {desc}?\n\n'
 'Answer LOW if the explanation says it is absent, little, small, or not '
 'visible. An explicit statement that something is NOT present is LOW, not '
 'NONE.\n'
 'Answer MODERATE if the explanation calls it moderate, weak, or partial. '
 'Note that "weak" means MODERATE here, not LOW.\n'
 'Answer HIGH if the explanation calls it strong, large, clear, dominant or '
 'strongly concentrated.\n'
 'Answer NONE only if the explanation never refers to this at all, in any '
 'form.\n\n'
 'Read the whole explanation and answer about this quantity only. Other '
 'quantities are described in the same text; do not answer about those.\n\n'
 'Reply with exactly one word: LOW, MODERATE, HIGH or NONE.\nAnswer:')

VERDICT_TMPL=(
 'An automated classifier proposed this diagnosis: {proposed}\n\n'
 'An analyst wrote this explanation of the vibration signal.\n\n'
 '---\n{body}\n---\n\n'
 'Does the explanation disagree with the proposed diagnosis?\n'
 'AGREE if it supports the proposal, DISAGREE if it argues for a different '
 'condition, UNCLEAR if it takes no position.\n\n'
 'Reply with exactly one word: AGREE, DISAGREE or UNCLEAR.\nAnswer:')

CLASS_TMPL=(
 'An analyst wrote this explanation of a bearing vibration signal.\n\n'
 '---\n{body}\n---\n\n'
 'Which condition does the explanation itself CONCLUDE?\n'
 'If it disagrees with a proposed diagnosis it will name the rejected one as '
 'well; report the one it concludes, which follows "consistent with".\n\n'
 'Reply with exactly one word: HEALTHY, INNER, OUTER or NONE.\nAnswer:')

LEVEL_VOCAB={'LOW':'low','MODERATE':'mid','MID':'mid','HIGH':'high','NONE':None}
VERDICT_VOCAB={'AGREE':'AGREE','DISAGREE':'DISAGREE','UNCLEAR':'UNCLEAR'}
CLASS_VOCAB={'HEALTHY':'Healthy','INNER':'Inner_Ring_Fault',
             'OUTER':'Outer_Ring_Fault','NONE':None}
PRETTY_SUP={'Healthy':'healthy bearing','Inner_Ring_Fault':'inner ring fault',
            'Outer_Ring_Fault':'outer ring fault'}

def first_word(text, vocab):
    t=(text or '').strip().upper()
    for w in re.split(r'[^A-Z]+', t):
        if w in vocab: return vocab[w], w
    return 'UNPARSED', None

_t=[('LOW','low'),('  moderate.','mid'),('HIGH\nbecause','high'),('NONE',None),
    ('banana','UNPARSED')]
assert all(first_word(a,LEVEL_VOCAB)[0]==b for a,b in _t), 'parser self-test failed'
print('parser self-test passed')

CANARY=[('No harmonic banding is visible at twice and three times that rate.',
         'env_bpfo_harmonic_ratio','low'),
        ('Weak harmonic content is present at twice and three times that rate.',
         'env_bpfo_harmonic_ratio','mid'),
        ('Clear harmonic banding is present at twice and three times that rate.',
         'env_bpfo_harmonic_ratio','high'),
        ('No shaft-rate sideband is present below that rate.',
         'env_bpfi_sideband_ratio','low'),
        ('A weak shaft-rate sideband is present below that rate.',
         'env_bpfi_sideband_ratio','mid'),
        ('A clear shaft-rate sideband appears below that rate.',
         'env_bpfi_sideband_ratio','high'),
        ('There is little energy at the inner-race passing rate near 123.7 Hz.',
         'env_bpfi_ratio','low'),
        ('There is moderate energy at the inner-race passing rate near 123.7 Hz.',
         'env_bpfi_ratio','mid'),
        ('Envelope energy is strongly concentrated at the inner-race passing '
         'rate near 123.7 Hz.','env_bpfi_ratio','high'),
        ('There is little energy at the outer-race passing rate near 76.3 Hz.',
         'env_bpfo_ratio','low'),
        ('Envelope energy is strongly concentrated at the outer-race passing '
         'rate near 76.3 Hz.','env_bpfo_ratio','high'),
        ('Only a small fraction of the envelope energy in the 20-500 Hz band '
         'falls at the bearing characteristic frequencies.',
         'env_char_energy_frac','low'),
        ('A large fraction of the envelope energy in the 20-500 Hz band is '
         'concentrated at the bearing characteristic frequencies.',
         'env_char_energy_frac','high')]
print(f'{len(CANARY)} canary sentences ready')

In [ ]:
#@title Load the judge
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
tok=AutoTokenizer.from_pretrained(JUDGE_MODEL); tok.padding_side='left'
if tok.pad_token is None: tok.pad_token=tok.eos_token
judge=AutoModelForCausalLM.from_pretrained(JUDGE_MODEL, dtype=torch.bfloat16,
                                           device_map='cuda:0').eval()
print(f'loaded {JUDGE_MODEL}: {sum(p.numel() for p in judge.parameters())/1e9:.1f}B, '
      f'{torch.cuda.memory_allocated()/2**30:.1f} GB allocated, '
      f'{torch.cuda.mem_get_info()[0]/2**30:.1f} GB free')

def build(user):
    return tok.apply_chat_template(
        [{'role':'system','content':JUDGE_SYS},{'role':'user','content':user}],
        tokenize=False, add_generation_prompt=True)

def ask(prompts, tag=''):
    outs=[]
    for i in range(0,len(prompts),JUDGE_BATCH):
        ch=prompts[i:i+JUDGE_BATCH]
        enc=tok(ch,return_tensors='pt',padding=True,truncation=True,
                max_length=2048).to('cuda:0')
        with torch.no_grad():
            o=judge.generate(**enc,max_new_tokens=JUDGE_MAX_NEW,do_sample=False,
                             pad_token_id=tok.eos_token_id)
        for k in range(len(ch)):
            outs.append(tok.decode(o[k][enc['input_ids'].shape[1]:],
                                   skip_special_tokens=True))
    return outs

In [ ]:
#@title Canary
DESC={q:d for q,d in QUANTS}
raw=ask([build(LEVEL_TMPL.format(body=f'<explanation>{s}</explanation>',
                                 desc=DESC[q])) for s,q,_ in CANARY])
per=defaultdict(lambda:[0,0]); print(f"{'expected':10}{'got':10}  sentence")
for (s,q,exp),r in zip(CANARY,raw):
    got,_=first_word(r,LEVEL_VOCAB)
    per[q][0]+=(got==exp); per[q][1]+=1
    print(f"{exp:10}{str(got):10}  {s[:56]}{'' if got==exp else '   <-- MISMATCH'}")
ok=sum(v[0] for v in per.values()); n=sum(v[1] for v in per.values())
print(f'\n{ok}/{n} correct')
for q,(a,b) in per.items(): print(f'  {q:26} {a}/{b}')
if ok<n:
    raise SystemExit('canary failed')

In [ ]:
#@title Judge every run
def cache_path(rid,cond):
    return CACHE/f'{rid}__{cond}__{JUDGE_MODEL.split("/")[-1]}.json'

t0=time.time(); done=0
for r in sorted(RUNS,key=lambda x:(x['dataset'],x['label'],x['modality'],x['ratio'],x['seed'])):
    for cond in r['conds']:
        k=(r['rid'],cond)
        if k not in DATA: continue
        cp=cache_path(r['rid'],cond)
        if cp.exists():
            print(f"{r['dataset']:4}{r['label']:20}{r['modality']:>6} r{r['ratio']:<4}s{r['seed']} {cond:13} cached"); continue
        rows=DATA[k]; idx=list(range(len(rows)))
        out={str(i):{} for i in idx}
        for q,desc in QUANTS:
            raw=ask([build(LEVEL_TMPL.format(body=rows[i]['body'],desc=desc))
                     for i in idx])
            for i,rr in zip(idx,raw): out[str(i)][q]=first_word(rr,LEVEL_VOCAB)[0]
        if cond in VERDICT_CONDS:
            raw=ask([build(VERDICT_TMPL.format(
                        proposed=PRETTY_SUP.get(rows[i]['supplied'],'none'),
                        body=rows[i]['body'])) for i in idx])
            for i,rr in zip(idx,raw):
                out[str(i)]['verdict']=first_word(rr,VERDICT_VOCAB)[0]
            raw=ask([build(CLASS_TMPL.format(body=rows[i]['body'])) for i in idx])
            for i,rr in zip(idx,raw):
                out[str(i)]['class']=first_word(rr,CLASS_VOCAB)[0]
        cp.write_text(json.dumps(out))
        done+=1
        el=(time.time()-t0)/60
        print(f"{r['dataset']:4}{r['label']:20}{r['modality']:>6} r{r['ratio']:<4}s{r['seed']} {cond:13} judged  "
              f"({el:.0f} min elapsed)")
print(f'\n{done} newly judged, {(time.time()-t0)/60:.1f} min')
del judge; gc.collect(); torch.cuda.empty_cache()

JUDGED={}
for r in RUNS:
    for cond in r['conds']:
        cp=cache_path(r['rid'],cond)
        if cp.exists():
            JUDGED[(r['rid'],cond)]={int(k):v for k,v in
                                     json.loads(cp.read_text()).items()}
print(f'{len(JUDGED)} run/condition pairs available')